In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pandas as pd


In [2]:
# get paths for each snap file for each folder in experiment_path
def get_snaps_many(experiment_path):
    runs = !ls $experiment_path
    runs = np.sort(np.asarray(runs).astype(int))

    snaps = {}
    for run in runs:
        snaps[run] = !ls $experiment_path/$run/snap.40_*.h5part

    return runs, snaps

# get paths for snapshots in a folder
def get_snaps(path):
    snaps = !ls $path/snap.40_*.h5part
    return snaps

# convert global.30 in run_path to a dataframe
def get_glob_df(run_path):
    glob_path = run_path + '/global.30'
    return pd.read_csv(glob_path, sep=r"\s+", index_col=False)
    
# build a df storing time and path data about a snapshot
def get_snap_times(run, snap_path, Myr_per_NB):
    with h5py.File(snap_path, 'r') as f:
        
        df = pd.DataFrame()
        df['Step'] = sorted(f.keys(), key=lambda k: f[k].attrs['Time'])
        df['Time[NB]'] = [f[step].attrs['Time'] for step in df['Step']]
        df['Time[Myr]'] = df['Time[NB]'] * Myr_per_NB
        df['snap_path'] = snap_path
        df["snap_name"] = df["snap_path"].str.split("/").str[-1]
        df['run'] = run
        return df
        
# build a df storing time and path data about multipl snapshots
def get_run_snap_times(run, snaps, glob_df):
    Myr_per_NB = glob_df['TIME[Myr]'].iloc[-1] / glob_df['TIME[NB}'].iloc[-1]

    dfs = []
    for snap_path in snaps:
        dfs.append(get_snap_times(run, snap_path, Myr_per_NB))

    return pd.concat(dfs, ignore_index=True).sort_values("Time[Myr]")

# build a df storing time and path data about every snapshot in an experiment
def get_experiment_snap_times(experiment_path):
    runs, snaps = get_snaps_many(experiment_path)

    dfs = []
    for run in runs:
        run_path = f'{experiment_path}/{run}'
        glob_df = get_glob_df(run_path)

        dfs.append(get_run_snap_times(run, snaps[run], glob_df))

    return pd.concat(dfs, ignore_index=True).sort_values("Time[Myr]")
        
# find the step and snapshot thats closest to a list of times
def find_times_Myr(snap_df, run, times):
    mask = snap_df['run'] == run

    idx = np.searchsorted(snap_df.loc[mask, 'Time[Myr]'], times)    
    idx[idx >= len(snap_df[mask])] = len(snap_df[mask]) - 1
    
    return snap_df[mask].iloc[idx].reset_index(drop=True)

# build a df for a step in a snap file 
def get_step(snap_path, step):
    df = pd.DataFrame()

    with h5py.File(snap_path, 'r') as f:
        s = f[step]
        df["M"] = s["M"][:]
        df["NAM"] = s["NAM"][:]
        df["POT"] = s["POT"][:]
        df["Vx"] = s["V1"][:]
        df["Vy"] = s["V2"][:]
        df["Vz"] = s["V3"][:]
        df["X"] = s["X1"][:]
        df["Y"] = s["X2"][:]
        df["Z"] = s["X3"][:]
        df.attrs['Time'] = s.attrs["Time"] * Myr_per_Nbody

        return df
        
# plot a step in a snap file on the given ax
def plot_step(fig, ax, snap_path, step, Myr_per_Nbody, offset=200, label=None,
              hidexlbl=False, hideylbl=False, hidetitle=False):   
    
    stepdf = get_step(snap_path, step)
    #print(stepdf)
        
    #com_x = np.average(x, weights=m)
    #com_z = np.average(z, weights=m)

    com_x = np.median(stepdf["X"])
    com_y = np.median( stepdf["Y"])

    hist, xedges, yedges = np.histogram2d(stepdf['X'] - com_x, stepdf['Y'] - com_y, bins=100,
                                          range=[[-offset, offset], [-offset, offset]])
    ax.imshow(
    hist.T,
    origin='lower',
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    cmap='gray_r',
    norm=LogNorm(),
    label=label
    )
    
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    
    if not hidetitle: ax.set_title(f'{stepdf.attrs['Time']:.1f} Myr')
    if not hidexlbl: ax.set_xlabel('X / pc')
    if not hideylbl: ax.set_ylabel('Y / pc')
    if label is not None: ax.text(-0.8*offset, 0.8*offset, label)
    ax.set_xlim(-offset, offset)
    ax.set_ylim(- offset, offset)
    ax.set_aspect('equal')    